# LSTM (법정동 단위, 통합 학습)

동마다 개별 LSTM을 학습하면 동당 데이터가 150~180개월 수준이라 과적합 위험이 큽니다. 대신 **62개 동을 하나의 모델로 묶어서** 학습하고, 동 임베딩으로 동별 가격 수준 차이를 반영합니다.

- 입력: 과거 24개월(동별 Train 구간 평균/표준편차로 정규화) + 동 임베딩
- 출력: 향후 12개월 예측 (SARIMA/Prophet과 동일한 horizon)
- 결측월이 섞인 윈도우는 학습/예측에서 제외 (SARIMA/Prophet 때와 동일한 원칙)

## 0. 환경 / 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("GPU 사용 가능:", len(tf.config.list_physical_devices('GPU')) > 0)
print("(GPU 없어도 이 정도 데이터 규모는 CPU로 충분합니다)")

## 1. 데이터 로드 및 타겟 설정

In [ ]:
raw = pd.read_csv("raw_series_dong.csv", encoding="utf-8-sig")
raw['계약일'] = pd.to_datetime(raw['계약일'])

TARGET_COL = '평균평당가격'  # 평당가격 예측 (월세로 바꾸려면 '월세'로 변경)

print(raw.shape)
print("법정동 수:", raw['법정동코드'].nunique())

## 2. Train / Valid / Test 기간 정의 (SARIMA/Prophet과 동일)

In [ ]:
TRAIN_END = pd.Timestamp('2023-12-01')
VALID_START, VALID_END = pd.Timestamp('2024-01-01'), pd.Timestamp('2024-12-01')
TEST_START, TEST_END = pd.Timestamp('2025-01-01'), pd.Timestamp('2025-12-01')

WINDOW = 24    # 과거 24개월을 보고
HORIZON = 12   # 향후 12개월 예측

## 3. 평가지표 함수 (SARIMA/Prophet 노트북과 동일)

In [ ]:
def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan

    y_true, y_pred = y_true[mask], y_pred[mask]
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mae, mape

## 4. 동 인덱스 매핑 (임베딩용)

In [ ]:
dong_list = sorted(raw['법정동코드'].unique().tolist())
dong_to_idx = {code: i for i, code in enumerate(dong_list)}
num_dongs = len(dong_list)

print("동 개수:", num_dongs)

## 5. 동별 스케일링 통계 (Train 구간만으로 계산 → 리키지 방지)
자치구 중앙값을 Train만으로 계산했던 것과 같은 원칙입니다. 이 함수는 예측 시점(cutoff)마다 다시 호출해서, 항상 '그 시점까지의 데이터'만 사용하도록 합니다.

In [ ]:
def compute_dong_stats(raw, target_col, cutoff):
    stats = {}
    for dong_code, grp in raw.groupby('법정동코드'):
        train_vals = grp[grp['계약일'] <= cutoff][target_col].dropna()
        mean_ = train_vals.mean()
        std_ = train_vals.std()
        if not np.isfinite(std_) or std_ == 0:
            std_ = 1.0
        stats[dong_code] = (mean_, std_)
    return stats

## 6. 슬라이딩 윈도우 데이터셋 생성
결측월이 섞인 윈도우(입력 24개월 또는 출력 12개월 어느 쪽이든)는 제외합니다.

In [ ]:
def build_windows(raw, target_col, cutoff, dong_stats, window=WINDOW, horizon=HORIZON):
    X_seq, X_dong, Y = [], [], []

    for dong_code, grp in raw.groupby('법정동코드'):
        grp = grp[grp['계약일'] <= cutoff].sort_values('계약일')
        series = grp.set_index('계약일')[target_col].asfreq('MS')

        mean_, std_ = dong_stats[dong_code]
        scaled = ((series - mean_) / std_).values

        n = len(scaled)
        for start in range(0, n - window - horizon + 1):
            x = scaled[start:start + window]
            y = scaled[start + window: start + window + horizon]

            if np.isnan(x).any() or np.isnan(y).any():
                continue

            X_seq.append(x)
            X_dong.append(dong_to_idx[dong_code])
            Y.append(y)

    return np.array(X_seq), np.array(X_dong), np.array(Y)

## 7. 모델 정의

In [ ]:
def build_model(num_dongs, window=WINDOW, horizon=HORIZON, embed_dim=8):
    seq_input = keras.Input(shape=(window, 1), name='seq_input')
    dong_input = keras.Input(shape=(1,), name='dong_input')

    x = layers.LSTM(32)(seq_input)

    d = layers.Embedding(num_dongs, embed_dim)(dong_input)
    d = layers.Flatten()(d)

    merged = layers.Concatenate()([x, d])
    merged = layers.Dense(32, activation='relu')(merged)
    output = layers.Dense(horizon)(merged)

    model = keras.Model(inputs=[seq_input, dong_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

## 8. Valid(2024) 학습 — Train(~2023) 구간 윈도우로만 학습

In [ ]:
dong_stats_train = compute_dong_stats(raw, TARGET_COL, TRAIN_END)
X_seq_train, X_dong_train, Y_train = build_windows(raw, TARGET_COL, TRAIN_END, dong_stats_train)

print("학습 샘플 수:", X_seq_train.shape[0])

model_valid = build_model(num_dongs)
history = model_valid.fit(
    [X_seq_train.reshape(-1, WINDOW, 1), X_dong_train], Y_train,
    epochs=30, batch_size=64, validation_split=0.1, verbose=0
)

print("최종 train loss:", history.history['loss'][-1])
print("최종 val loss:", history.history['val_loss'][-1])

## 9. Valid(2024) 예측 및 평가
동마다 Train 구간의 마지막 24개월을 입력으로 넣어서 2024년 12개월을 예측합니다.

In [ ]:
def predict_future(raw, target_col, cutoff, model, dong_stats, window=WINDOW):
    rows = []

    for dong_code in dong_list:
        grp = raw[(raw['법정동코드'] == dong_code) & (raw['계약일'] <= cutoff)].sort_values('계약일')
        series = grp.set_index('계약일')[target_col].asfreq('MS')

        mean_, std_ = dong_stats[dong_code]
        scaled = ((series - mean_) / std_).values

        if len(scaled) < window or np.isnan(scaled[-window:]).any():
            continue

        x = scaled[-window:].reshape(1, window, 1)
        d = np.array([[dong_to_idx[dong_code]]])

        pred_scaled = model.predict([x, d], verbose=0)[0]
        pred = pred_scaled * std_ + mean_

        rows.append((dong_code, pred))

    return dict(rows)


def get_actual(dong_code, start_date, end_date, target_col):
    sub = (
        raw[(raw['법정동코드'] == dong_code) &
            (raw['계약일'] >= start_date) & (raw['계약일'] <= end_date)]
        .sort_values('계약일')
    )
    return sub.set_index('계약일')[target_col]


lstm_valid_preds = predict_future(raw, TARGET_COL, TRAIN_END, model_valid, dong_stats_train)

rows = []
for dong_code, pred in lstm_valid_preds.items():
    actual = get_actual(dong_code, VALID_START, VALID_END, TARGET_COL)
    for i, (date, actual_val) in enumerate(actual.items()):
        rows.append({
            '법정동코드': dong_code, '계약일': date, 'horizon': i + 1,
            'actual': actual_val,
            'lstm_pred': pred[i] if i < len(pred) else np.nan,
        })

lstm_valid_long = pd.DataFrame(rows)
lstm_valid_long['lstm_ae'] = np.abs(lstm_valid_long['actual'] - lstm_valid_long['lstm_pred'])
lstm_valid_long['lstm_se'] = (lstm_valid_long['actual'] - lstm_valid_long['lstm_pred']) ** 2
lstm_valid_long['lstm_ape'] = np.abs(
    (lstm_valid_long['actual'] - lstm_valid_long['lstm_pred']) / lstm_valid_long['actual']
) * 100

print("LSTM 예측 성공 동 개수:", len(lstm_valid_preds), "/", len(dong_list))
print("2024 RMSE:", np.sqrt(lstm_valid_long['lstm_se'].mean()))
print("2024 MAE:", lstm_valid_long['lstm_ae'].mean())
print("2024 MAPE:", lstm_valid_long['lstm_ape'].mean())

## 10. Train+Valid로 재학습 → Test(2025) 평가

In [ ]:
dong_stats_full = compute_dong_stats(raw, TARGET_COL, VALID_END)
X_seq_full, X_dong_full, Y_full = build_windows(raw, TARGET_COL, VALID_END, dong_stats_full)

model_test = build_model(num_dongs)
model_test.fit(
    [X_seq_full.reshape(-1, WINDOW, 1), X_dong_full], Y_full,
    epochs=30, batch_size=64, validation_split=0.1, verbose=0
)

lstm_test_preds = predict_future(raw, TARGET_COL, VALID_END, model_test, dong_stats_full)

rows = []
for dong_code, pred in lstm_test_preds.items():
    actual = get_actual(dong_code, TEST_START, TEST_END, TARGET_COL)
    for i, (date, actual_val) in enumerate(actual.items()):
        rows.append({
            '법정동코드': dong_code, '계약일': date, 'horizon': i + 1,
            'actual': actual_val,
            'lstm_pred': pred[i] if i < len(pred) else np.nan,
        })

lstm_test_long = pd.DataFrame(rows)
lstm_test_long['lstm_ae'] = np.abs(lstm_test_long['actual'] - lstm_test_long['lstm_pred'])
lstm_test_long['lstm_se'] = (lstm_test_long['actual'] - lstm_test_long['lstm_pred']) ** 2
lstm_test_long['lstm_ape'] = np.abs(
    (lstm_test_long['actual'] - lstm_test_long['lstm_pred']) / lstm_test_long['actual']
) * 100

print("LSTM 예측 성공 동 개수:", len(lstm_test_preds), "/", len(dong_list))
print("2025 RMSE:", np.sqrt(lstm_test_long['lstm_se'].mean()))
print("2025 MAE:", lstm_test_long['lstm_ae'].mean())
print("2025 MAPE:", lstm_test_long['lstm_ape'].mean())

## 11. SARIMA/Prophet 결과와 합쳐서 비교 (선택)
SARIMA/Prophet 노트북에서 저장한 `sarima_prophet_valid_long.csv` / `sarima_prophet_test_long.csv`가 있으면 병합해서 세 모델을 한 표에서 비교할 수 있습니다.

In [ ]:
try:
    prev_valid = pd.read_csv("sarima_prophet_valid_long.csv", encoding="utf-8-sig")
    prev_valid['계약일'] = pd.to_datetime(prev_valid['계약일'])

    combined_valid = prev_valid.merge(
        lstm_valid_long[['법정동코드', '계약일', 'horizon', 'lstm_pred', 'lstm_ae', 'lstm_se', 'lstm_ape']],
        on=['법정동코드', '계약일', 'horizon'], how='inner'
    )

    print("=== 2024 전체 비교 ===")
    for prefix in ['sarima', 'prophet', 'naive', 'lstm']:
        rmse = np.sqrt(combined_valid[f'{prefix}_se'].mean())
        mape = combined_valid[f'{prefix}_ape'].mean()
        print(f"{prefix}: RMSE={rmse:.3f}, MAPE={mape:.2f}%")
except FileNotFoundError:
    print("sarima_prophet_valid_long.csv 파일을 찾을 수 없어 비교를 건너뜁니다.")

## 12. 저장

In [ ]:
lstm_valid_long.to_csv("lstm_valid_long.csv", index=False, encoding="utf-8-sig")
lstm_test_long.to_csv("lstm_test_long.csv", index=False, encoding="utf-8-sig")

## 13. 최종 배포용 모델 학습 및 저장 (평당가격 대시보드용)
지금까지는 Train/Valid/Test로 나눠서 **방법론 검증**을 했습니다. 실제 대시보드에 쓸 모델은 **가진 데이터 전부**를 써서 다시 학습하는 게 맞습니다 (평가용 모델과 배포용 모델은 다른 모델입니다).

여기서 저장하는 4개 파일(`lstm_model.keras`, `dong_stats.json`, `dong_to_idx.json`, `dong_info.csv`)과 `raw_series_dong.csv`를 Streamlit 앱과 같은 폴더에 두면 됩니다.

In [ ]:
import json

DEPLOY_TARGET_COL = '평균평당가격'  # 대시보드에서 예측할 값
MAX_DATE = raw['계약일'].max()
print("배포용 모델 학습에 사용하는 마지막 관측월:", MAX_DATE)

dong_stats_deploy = compute_dong_stats(raw, DEPLOY_TARGET_COL, MAX_DATE)
X_seq_deploy, X_dong_deploy, Y_deploy = build_windows(raw, DEPLOY_TARGET_COL, MAX_DATE, dong_stats_deploy)

print("배포용 학습 샘플 수:", X_seq_deploy.shape[0])

model_deploy = build_model(num_dongs)
model_deploy.fit(
    [X_seq_deploy.reshape(-1, WINDOW, 1), X_dong_deploy], Y_deploy,
    epochs=30, batch_size=64, validation_split=0.1, verbose=0
)

model_deploy.save("lstm_model.keras")

dong_stats_json = {
    str(int(k)): [float(v[0]), float(v[1])] for k, v in dong_stats_deploy.items()
}
with open("dong_stats.json", "w", encoding="utf-8-sig") as f:
    json.dump(dong_stats_json, f, ensure_ascii=False)

dong_to_idx_json = {str(int(k)): v for k, v in dong_to_idx.items()}
with open("dong_to_idx.json", "w", encoding="utf-8-sig") as f:
    json.dump(dong_to_idx_json, f, ensure_ascii=False)

dong_info = raw[['법정동코드', '법정동명', '자치구코드', '자치구명']].drop_duplicates()
dong_info.to_csv("dong_info.csv", index=False, encoding="utf-8-sig")

print("저장 완료: lstm_model.keras, dong_stats.json, dong_to_idx.json, dong_info.csv")
print("이 4개 파일 + raw_series_dong.csv를 Streamlit 앱과 같은 폴더에 두세요.")

## 13-1. 예측 가능한 동만 필터링 (대시보드 드롭다운용)

최근 `WINDOW`(24)개월 안에 결측월이 있거나 데이터 자체가 짧은 동은 예측이 안 됩니다. 이런 동을 대시보드 목록에서 아예 빼기 위해, `dong_info.csv`를 **예측 가능한 동만** 남기고 다시 저장합니다.

In [ ]:
predictable_dongs = []
excluded_dongs = []

for dong_code in dong_list:
    grp = raw[(raw['법정동코드'] == dong_code) & (raw['계약일'] <= MAX_DATE)].sort_values('계약일')
    series = grp.set_index('계약일')[DEPLOY_TARGET_COL].asfreq('MS')

    mean_, std_ = dong_stats_deploy[dong_code]
    scaled = ((series - mean_) / std_).values

    if len(scaled) >= WINDOW and not np.isnan(scaled[-WINDOW:]).any():
        predictable_dongs.append(dong_code)
    else:
        excluded_dongs.append(dong_code)

print(f"예측 가능한 동: {len(predictable_dongs)} / {len(dong_list)}")

if excluded_dongs:
    excluded_info = dong_info[dong_info['법정동코드'].isin(excluded_dongs)]
    print("제외된 동:")
    print(excluded_info[['자치구명', '법정동명']].to_string(index=False))

# dong_info.csv를 예측 가능한 동만 남기고 다시 저장 (앱은 이 파일을 그대로 읽으므로 코드 수정 불필요)
dong_info_filtered = dong_info[dong_info['법정동코드'].isin(predictable_dongs)]
dong_info_filtered.to_csv("dong_info.csv", index=False, encoding="utf-8-sig")

print("\ndong_info.csv 재저장 완료 (행 수:", dong_info_filtered.shape[0], ")")

## 14. 팀원 전달용 파일 (.h5 모델 + scaler.pkl)

**주의**: `scaler.pkl`을 나중에 다른 사람이 `joblib.load('scaler.pkl')`로 불러오려면, 그 사람 코드에도 아래 `DongScaler` 클래스 정의가 먼저 있어야 합니다 (없으면 `AttributeError: Can't get attribute 'DongScaler'` 에러가 납니다). 클래스 코드도 같이 전달하세요.

In [ ]:
import joblib


class DongScaler:
    """동(법정동코드)마다 별도의 평균/표준편차를 갖는 스케일러.
    sklearn의 StandardScaler와 달리, 동마다 절대 가격 수준이 크게 달라서
    전체를 하나로 스케일링하지 않고 법정동코드별로 따로 정규화한다.
    """

    def __init__(self, dong_stats):
        # dong_stats: {법정동코드(int): (mean, std)}
        self.dong_stats = dong_stats

    def transform(self, dong_code, values):
        mean_, std_ = self.dong_stats[int(dong_code)]
        return (np.asarray(values) - mean_) / std_

    def inverse_transform(self, dong_code, values):
        mean_, std_ = self.dong_stats[int(dong_code)]
        return np.asarray(values) * std_ + mean_


scaler = DongScaler(dong_stats_deploy)
joblib.dump(scaler, "scaler.pkl")

model_deploy.save("lstm_model.h5")

print("저장 완료: lstm_model.h5, scaler.pkl")
print("팀원에게는 이 두 파일 + 아래 정보를 같이 전달하세요.")
print()
print("WINDOW (과거 몇 개월을 입력으로 쓰는지):", WINDOW)
print("HORIZON (한 번에 예측하는 개월 수):", HORIZON)
print("동 임베딩 인덱스 매핑: dong_to_idx.json 참고")